# AAAI 2024

In [2]:
import sqlite3
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re 


In [5]:
import sqlite3
import requests
from bs4 import BeautifulSoup
import pandas as pd

def create_database(DB_PATH):
    """Create the database and the Conference table."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute('''
    CREATE TABLE IF NOT EXISTS Conference (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        Title TEXT NOT NULL,
        Author TEXT NOT NULL,
        PDF_Link TEXT,
        Code_URL TEXT,
        Conference_Name TEXT NOT NULL
    )
    ''')

    print("Conference 테이블이 생성되었습니다.")
    conn.commit()
    conn.close()

def save_to_database(df, conference_name, DB_PATH):
    conn = sqlite3.connect(DB_PATH, timeout=10)
    cursor = conn.cursor()

    try:
        for _, row in df.iterrows():  # ✅ iterrows() 사용하여 DataFrame의 각 행을 처리
            # 중복 데이터 확인
            cursor.execute('''
            SELECT 1 FROM Conference WHERE Title = ? AND Author = ? AND Conference_Name = ?
            ''', (row['title'], row['authors'], conference_name))
            result = cursor.fetchone()

            if not result:
                cursor.execute('''
                INSERT INTO Conference (Title, Author, PDF_Link, Code_URL, Conference_Name)
                VALUES (?, ?, ?, ?, ?)
                ''', (row['title'], row['authors'], row['pdf_link'], row['code_url'], conference_name))

        conn.commit()
        print(f"{len(df)}개의 논문이 {conference_name}에 저장되었습니다.")
    except sqlite3.Error as e:
        print(f"Database error: {e}")
    finally:
        conn.close()

In [46]:
def get_www_papers(html_file_path, conference_name):
    """저장된 HTML 파일에서 AAAI 학회의 Accepted Papers 정보를 크롤링 후 DataFrame 반환"""

    # HTML 파일 읽기
    with open(html_file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    papers = []

    # 논문 리스트가 포함된 테이블 찾기
    table = soup.find("table")

    if table:
        for row in table.find_all("tr")[1:]:  # 첫 번째 행(헤더) 제외
            columns = row.find_all("td")
            if len(columns) >= 3:
                # 제목과 링크 추출
                title_tag = columns[1].find("a")
                title_text = title_tag.text.strip() if title_tag else "Unknown"
                pdf_link = title_tag["href"] if title_tag and title_tag.has_attr("href") else None

                # 저자 정보 추출
                authors_raw = columns[2].text.strip()
                authors_cleaned = re.sub(r"\(.*?\)", "", authors_raw)  # 기관 정보 제거
                authors_cleaned = re.sub(r";\s*\n", ", ", authors_cleaned)  # ;\n 제거 후 , 로 변환
                authors_cleaned = re.sub(r"\s+", " ", authors_cleaned).strip()  # 불필요한 공백 정리

                papers.append({
                    "title": title_text,
                    "authors": authors_cleaned,
                    "pdf_link": pdf_link,  # PDF 링크
                    "code_url": None,
                    "conference_name": conference_name
                })

    # DataFrame으로 변환
    df_papers = pd.DataFrame(papers)
    return df_papers

In [42]:
url = 'https://www.paperdigest.org/2024/02/aaai-2024-papers-highlights/'
DB_PATH = "con_db/AAAI_conference_2024.db"
conference_name = 'AAAI 2024'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [43]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/AAAI_2024_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [47]:
df_papers = get_www_papers('html/AAAI_2024_accepted_papers.html', conference_name)

In [48]:
df_papers.head(10)

,title,authors,pdf_link,code_url,conference_name
0,T2I-Adapter: Learning Adapters to Dig Out More...,"Chong Mou , Xintao Wang , Liangbin Xie , Yanze...",https://www.paperdigest.org/paper/?paper_id=aa...,None,AAAI 2024
1,MemoryBank: Enhancing Large Language Models wi...,"Wanjun Zhong , Lianghong Guo , Qiqi Gao , He Y...",https://www.paperdigest.org/paper/?paper_id=aa...,None,AAAI 2024
2,Learning Temporal Resolution in Spectrogram fo...,"Haohe Liu , Xubo Liu , Qiuqiang Kong , Wenwu W...",https://www.paperdigest.org/paper/?paper_id=aa...,None,AAAI 2024
3,Machine-Created Universal Language for Cross-L...,"Yaobo Liang , Quanzhi Zhu , Junhe Zhao , Nan D...",https://www.paperdigest.org/paper/?paper_id=aa...,None,AAAI 2024
4,I Prefer Not to Say: Protecting User Consent i...,"Tobias Leemann , Martin Pawelczyk , Christian ...",https://www.paperdigest.org/paper/?paper_id=aa...,None,AAAI 2024
5,Investigating The Effectiveness of Task-Agnost...,"Seonghyeon Ye , Hyeonbin Hwang , Sohee Yang , ...",https://www.paperdigest.org/paper/?paper_id=aa...,None,AAAI 2024
6,ImageCaptioner2: Image Captioner for Image Cap...,"Eslam Abdelrahman , Pengzhan Sun , Li Erran Li...",https://www.paperdigest.org/paper/?paper_id=aa...,None,AAAI 2024
7,Preference Ranking Optimization for Human Alig...,"Feifan Song , Bowen Yu , Minghao Li , Haiyang ...",https://www.paperdigest.org/paper/?paper_id=aa...,None,AAAI 2024
8,Visual Adversarial Examples Jailbreak Aligned ...,"Xiangyu Qi , Kaixuan Huang , Ashwinee Panda , ...",https://www.paperdigest.org/paper/?paper_id=aa...,None,AAAI 2024
9,Leveraging Diffusion Perturbations for Measuri...,"Nicholas Lui , Bryan Chia , William Berrios , ...",https://www.paperdigest.org/paper/?paper_id=aa...,None,AAAI 2024


In [49]:
save_to_database(df_papers, conference_name, DB_PATH)

500개의 논문이 AAAI 2024에 저장되었습니다.


# 2023

In [50]:
url = 'https://www.paperdigest.org/2023/06/aaai-2023-highlights/'
DB_PATH = "con_db/AAAI_conference_2023.db"
conference_name = 'AAAI 2023'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [51]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/AAAI_2023_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [52]:
df_papers = get_www_papers('html/AAAI_2023_accepted_papers.html', conference_name)

In [53]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Back to The Future: Toward A Hybrid Architectu...,"Hasra Dodampegama , Mohan Sridharan ;",https://www.paperdigest.org/paper/?paper_id=aa...,None,AAAI 2023
1,Reducing ANN-SNN Conversion Error Through Resi...,"Zecheng Hao , Tong Bu , Jianhao Ding , Tiejun ...",https://www.paperdigest.org/paper/?paper_id=aa...,None,AAAI 2023
2,Hierarchical ConViT with Attention-Based Relat...,"Wentao He , Jialu Zhang , Jianfeng Ren , Ruibi...",https://www.paperdigest.org/paper/?paper_id=aa...,None,AAAI 2023
3,Deep Spiking Neural Networks with High Represe...,"Liwei Huang , Zhengyu Ma , Liutao Yu , Huihui ...",https://www.paperdigest.org/paper/?paper_id=aa...,None,AAAI 2023
4,A Semi-parametric Model for Decision Making in...,"Stephen Keeley , Benjamin Letham , Craig Sande...",https://www.paperdigest.org/paper/?paper_id=aa...,None,AAAI 2023


In [54]:
save_to_database(df_papers, conference_name, DB_PATH)

1720개의 논문이 AAAI 2023에 저장되었습니다.


# 2022 Virtual Page

In [59]:
url = 'https://dblp.org/db/conf/aaai/aaai2022.html'
DB_PATH = "con_db/AAAI_conference_2022.db"
conference_name = 'AAAI 2022'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [60]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/AAAI_2022_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [67]:
def get_www_papers(html_file_path, conference_name):
    """DBLP HTML 파일에서 논문 제목과 저자 정보 추출"""
    
    with open(html_file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    papers = []
    
    # 세션 정보 추출을 위한 변수
    current_session = "Main Track"
    
    # 모든 논문 항목 처리
    for entry in soup.select('ul.publ-list li.entry'):
        # 세션 정보 업데이트 (h2 태그가 있는 경우)
        prev_element = entry.find_previous_sibling()
        if prev_element and prev_element.name == 'h2':
            current_session = prev_element.text.strip()
            
        # 제목 추출
        title_tag = entry.select_one('span.title')
        title = title_tag.text.strip() if title_tag else "No Title"
        
        # 저자 추출 (모든 저자 태그 수집)
        authors = [author.text.strip() for author in entry.select('span[itemprop="author"] a')]
        
        # PDF 링크 추출 (존재하는 경우)
        pdf_tag = entry.select_one('nav.publ ul li.drop-down a[title="pdf"]')
        pdf_link = pdf_tag['href'] if pdf_tag else None
        
        papers.append({
            "title": title,
            "authors": ", ".join(authors),
            "pdf_link": pdf_link,
            'code_url': None,
            "conference_name": conference_name
        })
    
    return pd.DataFrame(papers)

In [68]:
df_papers = get_www_papers('html/AAAI_2022_accepted_papers.html', conference_name)

In [71]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
1,Learning Unseen Emotions from Gestures via Sem...,"Abhishek Banerjee, Uttaran Bhattacharya, Anike...",None,None,AAAI 2022
2,Optimized Potential Initialization for Low-Lat...,"Tong Bu, Jianhao Ding, Zhaofei Yu, Tiejun Huang",None,None,AAAI 2022
3,Planning with Biological Neurons and Synapses.,"Francesco D'Amore, Daniel Mitropolsky, Pierlui...",None,None,AAAI 2022
4,Backprop-Free Reinforcement Learning with Acti...,"Alexander G. Ororbia II, Ankur Arjun Mali",None,None,AAAI 2022
5,VECA: A New Benchmark and Toolkit for General ...,"Kwanyoung Park, Hyunseok Oh, Youngki Lee",None,None,AAAI 2022


In [72]:
save_to_database(df_papers, conference_name, DB_PATH)

1624개의 논문이 AAAI 2022에 저장되었습니다.


# 2021

In [74]:
url = 'https://dblp.org/db/conf/aaai/aaai2021.html'
DB_PATH = "con_db/AAAI_conference_2021.db"
conference_name = 'AAAI 2021'
create_database(DB_PATH)

CSV 파일이 성공적으로 생성되었습니다: aaai_2021_papers.csv


In [77]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/AAAI_2021_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [78]:
df_papers = get_www_papers('html/AAAI_2021_accepted_papers.html', conference_name)

In [79]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Thirty-Sixth AAAI Conference on Artificial Int...,,None,None,AAAI 2022
1,Learning Unseen Emotions from Gestures via Sem...,"Abhishek Banerjee, Uttaran Bhattacharya, Anike...",None,None,AAAI 2022
2,Optimized Potential Initialization for Low-Lat...,"Tong Bu, Jianhao Ding, Zhaofei Yu, Tiejun Huang",None,None,AAAI 2022
3,Planning with Biological Neurons and Synapses.,"Francesco D'Amore, Daniel Mitropolsky, Pierlui...",None,None,AAAI 2022
4,Backprop-Free Reinforcement Learning with Acti...,"Alexander G. Ororbia II, Ankur Arjun Mali",None,None,AAAI 2022


In [80]:
df_papers = df_papers.drop(index=[0])

In [81]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
1,Learning Unseen Emotions from Gestures via Sem...,"Abhishek Banerjee, Uttaran Bhattacharya, Anike...",None,None,AAAI 2022
2,Optimized Potential Initialization for Low-Lat...,"Tong Bu, Jianhao Ding, Zhaofei Yu, Tiejun Huang",None,None,AAAI 2022
3,Planning with Biological Neurons and Synapses.,"Francesco D'Amore, Daniel Mitropolsky, Pierlui...",None,None,AAAI 2022
4,Backprop-Free Reinforcement Learning with Acti...,"Alexander G. Ororbia II, Ankur Arjun Mali",None,None,AAAI 2022
5,VECA: A New Benchmark and Toolkit for General ...,"Kwanyoung Park, Hyunseok Oh, Youngki Lee",None,None,AAAI 2022


In [82]:
save_to_database(df_papers, conference_name, DB_PATH)

1624개의 논문이 AAAI 2022에 저장되었습니다.


# 2020

In [83]:
url = 'https://dblp.org/db/conf/aaai/aaai2020.html'
DB_PATH = "con_db/AAAI_conference_2020.db"
conference_name = 'AAAI 2020'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [84]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/AAAI_2020_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [85]:
df_papers = get_www_papers('html/AAAI_2020_accepted_papers.html', conference_name)

In [86]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,The Thirty-Fourth AAAI Conference on Artificia...,,None,None,AAAI 2020
1,Balancing Spreads of Influence in a Social Net...,"Ruben Becker, Federico Corò, Gianlorenzo D'Ang...",None,None,AAAI 2020
2,MultiSumm: Towards a Unified Model for Multi-L...,"Yue Cao, Xiaojun Wan, Jin-ge Yao, Dian Yu",None,None,AAAI 2020
3,Efficient Heterogeneous Collaborative Filterin...,"Chong Chen, Min Zhang, Yongfeng Zhang, Weizhi ...",None,None,AAAI 2020
4,Revisiting Graph Based Collaborative Filtering...,"Lei Chen, Le Wu, Richang Hong, Kun Zhang, Meng...",None,None,AAAI 2020


In [87]:
df_papers = df_papers.drop(index=[0])

In [88]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
1,Balancing Spreads of Influence in a Social Net...,"Ruben Becker, Federico Corò, Gianlorenzo D'Ang...",None,None,AAAI 2020
2,MultiSumm: Towards a Unified Model for Multi-L...,"Yue Cao, Xiaojun Wan, Jin-ge Yao, Dian Yu",None,None,AAAI 2020
3,Efficient Heterogeneous Collaborative Filterin...,"Chong Chen, Min Zhang, Yongfeng Zhang, Weizhi ...",None,None,AAAI 2020
4,Revisiting Graph Based Collaborative Filtering...,"Lei Chen, Le Wu, Richang Hong, Kun Zhang, Meng...",None,None,AAAI 2020
5,Question-Driven Purchasing Propensity Analysis...,"Long Chen, Ziyu Guan, Qibin Xu, Qiong Zhang, H...",None,None,AAAI 2020


In [89]:
save_to_database(df_papers, conference_name, DB_PATH)

1864개의 논문이 AAAI 2020에 저장되었습니다.


# 2019

In [90]:
url = 'https://dblp.org/db/conf/aaai/aaai2019.html'
DB_PATH = "con_db/AAAI_conference_2019.db"
conference_name = 'AAAI 2019'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [91]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/AAAI_2019_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [92]:
df_papers = get_www_papers('html/AAAI_2019_accepted_papers.html', conference_name)

In [93]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,The Thirty-Third AAAI Conference on Artificial...,,None,None,AAAI 2019
1,Incorporating Behavioral Constraints in Online...,"Avinash Balakrishnan, Djallel Bouneffouf, Nich...",None,None,AAAI 2019
2,Outlier Aware Network Embedding for Attributed...,"Sambaran Bandyopadhyay, Lokesh Nagalapatti, M....",None,None,AAAI 2019
3,Comparative Document Summarisation via Classif...,"Umanga Bista, Alexander Patrick Mathews, Minje...",None,None,AAAI 2019
4,ColNet: Embedding the Semantics of Web Tables ...,"Jiaoyan Chen, Ernesto Jiménez-Ruiz, Ian Horroc...",None,None,AAAI 2019


In [94]:
df_papers = df_papers.drop(index=[0])

In [95]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
1,Incorporating Behavioral Constraints in Online...,"Avinash Balakrishnan, Djallel Bouneffouf, Nich...",None,None,AAAI 2019
2,Outlier Aware Network Embedding for Attributed...,"Sambaran Bandyopadhyay, Lokesh Nagalapatti, M....",None,None,AAAI 2019
3,Comparative Document Summarisation via Classif...,"Umanga Bista, Alexander Patrick Mathews, Minje...",None,None,AAAI 2019
4,ColNet: Embedding the Semantics of Web Tables ...,"Jiaoyan Chen, Ernesto Jiménez-Ruiz, Ian Horroc...",None,None,AAAI 2019
5,Improving One-Class Collaborative Filtering vi...,"Jin Chen, Defu Lian, Kai Zheng",None,None,AAAI 2019


In [96]:
save_to_database(df_papers, conference_name, DB_PATH)

1343개의 논문이 AAAI 2019에 저장되었습니다.


# 2018

In [97]:
url = 'https://dblp.org/db/conf/aaai/aaai2018.html'
DB_PATH = "con_db/AAAI_conference_2018.db"
conference_name = 'AAAI 2018'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [98]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/AAAI_2018_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [99]:
df_papers = get_www_papers('html/AAAI_2018_accepted_papers.html', conference_name)

In [100]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Proceedings of the Thirty-Second AAAI Conferen...,"Sheila A. McIlraith, Kilian Q. Weinberger",None,None,AAAI 2018
1,Algorithms for Trip-Vehicle Assignment in Ride...,"Xiaohui Bei, Shengyu Zhang",None,None,AAAI 2018
2,EAD: Elastic-Net Attacks to Deep Neural Networ...,"Pin-Yu Chen, Yash Sharma, Huan Zhang, Jinfeng ...",None,None,AAAI 2018
3,Learning Differences Between Visual Scanning P...,"Jonathan Chung, Moshe Eizenman, Uros Rakita, R...",None,None,AAAI 2018
4,Comparing Population Means Under Local Differe...,"Bolin Ding, Harsha Nori, Paul Li, Joshua Allen",None,None,AAAI 2018


In [101]:
df_papers = df_papers.drop(index=[0]) 


In [102]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
1,Algorithms for Trip-Vehicle Assignment in Ride...,"Xiaohui Bei, Shengyu Zhang",None,None,AAAI 2018
2,EAD: Elastic-Net Attacks to Deep Neural Networ...,"Pin-Yu Chen, Yash Sharma, Huan Zhang, Jinfeng ...",None,None,AAAI 2018
3,Learning Differences Between Visual Scanning P...,"Jonathan Chung, Moshe Eizenman, Uros Rakita, R...",None,None,AAAI 2018
4,Comparing Population Means Under Local Differe...,"Bolin Ding, Harsha Nori, Paul Li, Joshua Allen",None,None,AAAI 2018
5,MuseGAN: Multi-track Sequential Generative Adv...,"Hao-Wen Dong, Wen-Yi Hsiao, Li-Chia Yang, Yi-H...",None,None,AAAI 2018


In [103]:
save_to_database(df_papers, conference_name, DB_PATH)

1102개의 논문이 AAAI 2018에 저장되었습니다.


# 2017

In [105]:
url = 'https://dblp.org/db/conf/aaai/aaai2017.html'
DB_PATH = "con_db/AAAI_conference_2017.db"
conference_name = 'AAAI 2017'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [106]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/AAAI_2017_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [107]:
df_papers = get_www_papers('html/AAAI_2017_accepted_papers.html', conference_name)

In [108]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Proceedings of the Thirty-First AAAI Conferenc...,"Satinder Singh, Shaul Markovitch",None,None,AAAI 2017
1,SnapNETS: Automatic Segmentation of Network Se...,"Sorour E. Amiri, Liangzhe Chen, B. Aditya Prakash",None,None,AAAI 2017
2,Taming the Matthew Effect in Online Markets wi...,"Franco Berbeglia, Pascal Van Hentenryck",None,None,AAAI 2017
3,A Leukocyte Detection Technique in Blood Smear...,"Deblina Bhattacharjee, Anand Paul",None,None,AAAI 2017
4,Partitioned Sampling of Public Opinions Based ...,"Weiran Huang, Liang Li, Wei Chen",None,None,AAAI 2017


In [109]:
df_papers = df_papers.drop(index=[0])

In [110]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
1,SnapNETS: Automatic Segmentation of Network Se...,"Sorour E. Amiri, Liangzhe Chen, B. Aditya Prakash",None,None,AAAI 2017
2,Taming the Matthew Effect in Online Markets wi...,"Franco Berbeglia, Pascal Van Hentenryck",None,None,AAAI 2017
3,A Leukocyte Detection Technique in Blood Smear...,"Deblina Bhattacharjee, Anand Paul",None,None,AAAI 2017
4,Partitioned Sampling of Public Opinions Based ...,"Weiran Huang, Liang Li, Wei Chen",None,None,AAAI 2017
5,Novel Geometric Approach for Global Alignment ...,"Yangwei Liu, Hu Ding, Danyang Chen, Jinhui Xu",None,None,AAAI 2017


In [111]:
save_to_database(df_papers, conference_name, DB_PATH)

786개의 논문이 AAAI 2017에 저장되었습니다.


# 2016

In [112]:
url = 'https://dblp.org/db/conf/aaai/aaai2016.html'
DB_PATH = "con_db/AAAI_conference_2016.db"
conference_name = 'AAAI 2016'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [113]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/AAAI_2016_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [114]:
df_papers = get_www_papers('html/AAAI_2016_accepted_papers.html', conference_name)

In [115]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Proceedings of the Thirtieth AAAI Conference o...,"Dale Schuurmans, Michael P. Wellman",None,None,AAAI 2016
1,Inferring Multi-Dimensional Ideal Points for U...,"Mohammad Raihanul Islam, K. S. M. Tozammel Hos...",None,None,AAAI 2016
2,Little Is Much: Bridging Cross-Platform Behavi...,"Meng Jiang, Peng Cui, Nicholas Jing Yuan, Xing...",None,None,AAAI 2016
3,Scientific Ranking over Heterogeneous Academic...,"Ronghua Liang, Xiaorui Jiang",None,None,AAAI 2016
4,MUST-CNN: A Multilayer Shift-and-Stitch Deep C...,"Zeming Lin, Jack Lanchantin, Yanjun Qi",None,None,AAAI 2016


In [116]:
df_papers = df_papers.drop(index=[0])

In [117]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
1,Inferring Multi-Dimensional Ideal Points for U...,"Mohammad Raihanul Islam, K. S. M. Tozammel Hos...",None,None,AAAI 2016
2,Little Is Much: Bridging Cross-Platform Behavi...,"Meng Jiang, Peng Cui, Nicholas Jing Yuan, Xing...",None,None,AAAI 2016
3,Scientific Ranking over Heterogeneous Academic...,"Ronghua Liang, Xiaorui Jiang",None,None,AAAI 2016
4,MUST-CNN: A Multilayer Shift-and-Stitch Deep C...,"Zeming Lin, Jack Lanchantin, Yanjun Qi",None,None,AAAI 2016
5,Hospital Stockpiling Problems with Inventory S...,"Eric Lofgren, Anil Vullikanti",None,None,AAAI 2016


In [118]:
save_to_database(df_papers, conference_name, DB_PATH)

691개의 논문이 AAAI 2016에 저장되었습니다.


# 2015

In [119]:
url = 'https://dblp.org/db/conf/aaai/aaai2015.html'
DB_PATH = "con_db/AAAI_conference_2015.db"
conference_name = 'AAAI 2015'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [120]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/AAAI_2015_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [121]:
df_papers = get_www_papers('html/AAAI_2015_accepted_papers.html', conference_name)

In [122]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Proceedings of the Twenty-Ninth AAAI Conferenc...,"Blai Bonet, Sven Koenig",None,None,AAAI 2015
1,Efficient Top-k Shortest-Path Distance Queries...,"Takuya Akiba, Takanori Hayashi, Nozomi Nori, Y...",None,None,AAAI 2015
2,Inferring Same-As Facts from Linked Data: An I...,"Mustafa Al-Bakri, Manuel Atencia, Steffen Lala...",None,None,AAAI 2015
3,A Personalized Interest-Forgetting Markov Mode...,"Jun Chen, Chaokun Wang, Jianmin Wang",None,None,AAAI 2015
4,"Will You ""Reconsume"" the Near Past? Fast Predi...","Jun Chen, Chaokun Wang, Jianmin Wang",None,None,AAAI 2015


In [123]:
df_papers = df_papers.drop(index=[0])

In [124]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
1,Efficient Top-k Shortest-Path Distance Queries...,"Takuya Akiba, Takanori Hayashi, Nozomi Nori, Y...",None,None,AAAI 2015
2,Inferring Same-As Facts from Linked Data: An I...,"Mustafa Al-Bakri, Manuel Atencia, Steffen Lala...",None,None,AAAI 2015
3,A Personalized Interest-Forgetting Markov Mode...,"Jun Chen, Chaokun Wang, Jianmin Wang",None,None,AAAI 2015
4,"Will You ""Reconsume"" the Near Past? Fast Predi...","Jun Chen, Chaokun Wang, Jianmin Wang",None,None,AAAI 2015
5,VELDA: Relating an Image Tweet's Text and Images.,"Tao Chen, Hany M. SalahEldeen, Xiangnan He, Mi...",None,None,AAAI 2015


In [125]:
save_to_database(df_papers, conference_name, DB_PATH)

674개의 논문이 AAAI 2015에 저장되었습니다.


# 2014

In [126]:
url = 'https://dblp.org/db/conf/aaai/aaai2014.html'
DB_PATH = "con_db/AAAI_conference_2014.db"
conference_name = 'AAAI 2014'
create_database(DB_PATH)

Conference 테이블이 생성되었습니다.


In [127]:
response = requests.get(url) 
soup =BeautifulSoup(response.text, 'html.parser')
with open('html/AAAI_2014_accepted_papers.html', 'w', encoding='utf-8') as f:
    f.write(soup.prettify())

In [128]:
df_papers = get_www_papers('html/AAAI_2014_accepted_papers.html', conference_name)

In [129]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
0,Proceedings of the Twenty-Eighth AAAI Conferen...,"Carla E. Brodley, Peter Stone",None,None,AAAI 2014
1,TopicMF: Simultaneously Exploiting Ratings and...,"Yang Bao, Hui Fang, Jie Zhang",None,None,AAAI 2014
2,Context-Aware Collaborative Topic Regression w...,"Chaochao Chen, Xiaolin Zheng, Yan Wang, Fuxing...",None,None,AAAI 2014
3,Improving Context and Category Matching for En...,"Yueguo Chen, Lexi Gao, Shuming Shi, Xiaoyong D...",None,None,AAAI 2014
4,Machine Translation with Real-Time Web Search.,"Lei Cui, Ming Zhou, Qiming Chen, Dongdong Zhan...",None,None,AAAI 2014


In [130]:
df_papers = df_papers.drop(index=[0])

In [131]:
df_papers.head()

,title,authors,pdf_link,code_url,conference_name
1,TopicMF: Simultaneously Exploiting Ratings and...,"Yang Bao, Hui Fang, Jie Zhang",None,None,AAAI 2014
2,Context-Aware Collaborative Topic Regression w...,"Chaochao Chen, Xiaolin Zheng, Yan Wang, Fuxing...",None,None,AAAI 2014
3,Improving Context and Category Matching for En...,"Yueguo Chen, Lexi Gao, Shuming Shi, Xiaoyong D...",None,None,AAAI 2014
4,Machine Translation with Real-Time Web Search.,"Lei Cui, Ming Zhou, Qiming Chen, Dongdong Zhan...",None,None,AAAI 2014
5,Leveraging Decomposed Trust in Probabilistic M...,"Hui Fang, Yang Bao, Jie Zhang",None,None,AAAI 2014


In [132]:
save_to_database(df_papers, conference_name, DB_PATH)

474개의 논문이 AAAI 2014에 저장되었습니다.
